# Stage 1 — DreamBooth + LoRA per [V]

In [ ]:
import torch, sys, os

print(f'Python:           {sys.version[:6]}')
print(f'PyTorch:          {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'GPU:              {torch.cuda.get_device_name(0)}')
    print(f'VRAM:      {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    raise RuntimeError('GPU not available')

In [ ]:
!pip install -q \
    diffusers==0.27.2 \
    transformers==4.40.0 \
    accelerate==0.29.3 \
    peft==0.10.0 \
    wandb==0.16.6 \
    torchmetrics==1.3.2 \
    Pillow tqdm pyyaml

In [ ]:
# Login HuggingFace and Wandb 

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
import wandb

secrets = UserSecretsClient()

# HuggingFace
hf_token = secrets.get_secret('HF_TOKEN')
login(token=hf_token)
print('HuggingFace: OK')

# Wandb
wandb_token = secrets.get_secret('WANDB_API_KEY')
wandb.login(key=wandb_token)
print('Wandb: OK')

In [ ]:
import os

WORK = '/kaggle/working'

for d in ['checkpoints/stage1', 'checkpoints/stage2',
          'generated', 'splits', 'logs']:
    os.makedirs(f'{WORK}/{d}', exist_ok=True)

REPO_URL = 'https://github.com/alecanc/Few-Shot-Personalization-of-a-Diffusion-Model-for-Industrial-Defect-Synthesis'
!git clone {REPO_URL} {WORK}/repo

os.chdir(f'{WORK}/repo')
print(f'Working directory: {os.getcwd()}')
print(os.listdir('.'))

In [ ]:
# Verify MVTec AD

from pathlib import Path

MVTEC = Path('/kaggle/input/mvtec-ad') 

for cat in ['bottle', 'metal_nut', 'leather']:
    cat_dir = MVTEC / cat
    if not cat_dir.exists():
        print(f'Missing: {cat_dir}')
        continue
    n_good = len(list((cat_dir / 'train/good').glob('*.png')))
    defect_types = [d.name for d in (cat_dir / 'test').iterdir() if d.name != 'good']
    print(f'{cat}: {n_good} clean | defect types: {defect_types}')

In [ ]:
import yaml

with open('config.yaml') as f:
    cfg = yaml.safe_load(f)

cfg['paths']['mvtec_root']  = str(MVTEC)
cfg['paths']['output_root'] = WORK
cfg['paths']['checkpoints'] = f'{WORK}/checkpoints'
cfg['paths']['splits_dir']  = f'{WORK}/splits'
cfg['paths']['logs']        = f'{WORK}/logs'

with open('config.yaml', 'w') as f:
    yaml.dump(cfg, f)

print('config.yaml aggiornato.')
print(yaml.dump(cfg['paths'], default_flow_style=False))

In [ ]:
# Generate split dataset

CATEGORY = 'bottle'  # to change for every category

!python data/splits.py --config config.yaml

# Verify splits
import json
with open(f'{WORK}/splits/{CATEGORY}_split.json') as f:
    split = json.load(f)

print(f"Stage 1: {len(split['stage1'])} images")
for dtype, imgs in split['stage2'].items():
    print(f"Stage 2 [{dtype}]: {len(imgs)} train | {len(split['eval'][dtype])} eval")

In [ ]:
# Generate prior images for Stage 1 training


!python train/generate_prior_images.py \
    --config config.yaml \
    --category {CATEGORY}

# Verify prior images
prior_dir = Path(f'{WORK}/checkpoints/stage1/{CATEGORY}/prior_images')
print(f' Prior images generated: {len(list(prior_dir.glob("*.png")))}')

In [ ]:
# Training Stage 1


!python train/train_stage1.py \
    --config config.yaml \
    --category {CATEGORY}

In [ ]:
final_dir = Path(f'{WORK}/checkpoints/stage1/{CATEGORY}/final')
print('File in\'adapter:')
for f in final_dir.iterdir():
    print(f'  {f.name} ({f.stat().st_size/1e6:.1f} MB)')

In [ ]:
import torch
from diffusers import StableDiffusionPipeline
import yaml

with open('config.yaml') as f:
    cfg = yaml.safe_load(f)

pipe = StableDiffusionPipeline.from_pretrained(
    cfg['model_id'],
    torch_dtype=torch.float16,
    safety_checker=None,
).to('cuda')

# Load LoRA weights
pipe.load_lora_weights(str(final_dir))

# Generate with the token [V]
token_V = cfg['token_V'] 
prompt = f'a photo of a {token_V} {CATEGORY}'

images = pipe(
    prompt,
    num_images_per_prompt=4,
    num_inference_steps=30,
    guidance_scale=7.5,
    generator=torch.Generator('cuda').manual_seed(42),
).images

# Visualize results
from PIL import Image as PILImage
grid = PILImage.new('RGB', (1024, 1024))
for i, img in enumerate(images):
    grid.paste(img, ((i%2)*512, (i//2)*512))
grid.save(f'{WORK}/stage1_test_output.png')
grid  

In [ ]:
# Save checkpoint

import shutil

export_dir = f'{WORK}/stage1_lora_{CATEGORY}_final'
shutil.copytree(str(final_dir), export_dir, dirs_exist_ok=True)

print(f'Checkpoint ready to\'export: {export_dir}')
